# The hold-out track record


# The hold-out track record

Every labelled gene was hidden once and each strategy that calls labels was asked what it is. This notebook reads the record that ships with Starplast and states what it says. It was built by `scripts/build_track_record.py` (`starplast/track_record.py`), design in `instructions/open/62_holdout_track_record.md`.

Three rules make it trustworthy rather than merely present: the five folds never split an orthogroup (a paralogue cannot give away its sibling's label); a strategy that says nothing is counted apart from one that is wrong; and no percentage is printed from fewer than five answered genes.

In [1]:
import pandas as pd
from starplast import track_record as T
led = pd.read_parquet(T.shipped_path())
led.groupby(['organism', 'mode'], observed=True).size().rename('rows')

organism  mode    
Pf        set         57176
          together    22120
Tg        set         58072
          together    53256

## Coverage

`together` rows are the fold hold-outs: one per (strategy, labelled gene). `set` rows hide several genes at once. Each fold-held gene must appear once per strategy -- no more, no less.

In [2]:
folds = led[led['mode'] == 'together']
per = folds.groupby(['organism', 'strategy'], observed=True)['gene'].agg(['size', 'nunique'])
assert (per['size'] == per['nunique']).all(), 'a gene was held out twice'
per.rename(columns={'size': 'held out', 'nunique': 'distinct genes'})

held out  distinct genes
organism strategy                                       
Pf       cluster_guilt              1580            1580
         feature_knn                1580            1580
         graph_convolution          1580            1580
         layer_propagation          1580            1580
         layer_vote                 1580            1580
         map_neighbours             1580            1580
         physical_partners          1580            1580
         random_forest              1580            1580
         stacking                   1580            1580
         stratum_focus              1580            1580
         structural_homology        1580            1580
         supervised_classifier      1580            1580
         triangulation              1580            1580
         understudied_first         1580            1580
Tg       cluster_guilt              3804            3804
         feature_knn                3804            3804
         graph_convolution          3804            3804
         layer_propagation          3804            3804
         layer_vote                 3804            3804
         map_neighbours             3804            3804

## Strategy level

Right over answered, with the Wilson 95% interval. `answered` below the number held out means the strategy abstained on the rest -- shown, not hidden in the rate.

In [3]:
s = T.summary(folds, 'strategy')
s[['organism', 'strategy', 'answered', 'right', 'wrong', 'abstained', 'rate', 'rate_low', 'rate_high']].sort_values(['organism', 'rate'], ascending=[True, False]).round(3)

,organism,strategy,answered,right,wrong,abstained,rate,rate_low,rate_high
6,Pf,physical_partners,58,48,10,1522,0.828,0.711,0.904
12,Pf,triangulation,1205,831,374,375,0.69,0.663,0.715
13,Pf,understudied_first,1205,831,374,375,0.69,0.663,0.715
7,Pf,random_forest,1580,1020,560,0,0.646,0.622,0.669
1,Pf,feature_knn,1264,777,487,316,0.615,0.588,0.641
9,Pf,stratum_focus,1264,777,487,316,0.615,0.588,0.641
2,Pf,graph_convolution,1580,938,642,0,0.594,0.569,0.618
8,Pf,stacking,1580,932,648,0,0.59,0.565,0.614
11,Pf,supervised_classifier,1580,906,674,0,0.573,0.549,0.598
4,Pf,layer_vote,1580,868,712,0,0.549,0.525,0.574


## Class level: which distinctions the data carries

For each class, the best strategy's rate. A class every strategy misses is a statement about the measurements, not about any one strategy.

In [4]:
c = T.summary(folds[folds.organism == 'Tg'], 'class')
c = c[c['enough'].astype(bool)]
best = c.sort_values('rate', ascending=False).drop_duplicates('truth')
best[['truth', 'strategy', 'right', 'answered', 'rate', 'confused_with']].round(3)

,truth,strategy,right,answered,rate,confused_with
144,19S proteasome,physical_partners,16,16,1,
148,ER 2,physical_partners,8,8,1,
146,60S ribosome,physical_partners,37,40,0.925,"nucleus - chromatin, nucleus - non-chromatin, cytosol"
150,IMC,physical_partners,34,37,0.919,"cytosol, PM - peripheral 2"
248,PM - peripheral 1,structural_homology,21,23,0.913,"IMC, nucleus - chromatin"
161,mitochondrion - membranes,physical_partners,54,61,0.885,"mitochondrion - soluble, nucleus - chromatin"
160,micronemes,physical_partners,21,24,0.875,"cytosol, ER 2"
158,dense granules,physical_partners,45,56,0.804,"cytosol, ER 2, mitochondrion - soluble"
46,rhoptries 1,feature_knn,36,46,0.783,"rhoptries 2, nucleus - chromatin, dense granules"
44,nucleus - chromatin,feature_knn,463,634,0.73,"nucleus - non-chromatin, cytosol, mitochondrion - soluble"


## Hidden together: does it still work when many genes are unknown?

Random sets of 1 to 500 genes hidden at once. A strategy that only works when a gene's neighbours are labelled falls as the set grows; a flat curve means the evidence reaches the genes themselves.

In [5]:
sets = led[(led['mode'] == 'set') & led['set_name'].astype(str).str.startswith('random')]
curve = sets.assign(answered=~sets['abstained'].astype(bool))
curve = curve[curve['answered']].groupby(['organism', 'strategy', 'set_size'], observed=True)['correct'].mean().unstack('set_size')
curve.round(2)

set_size                         1         5         20        100       500
organism strategy                                                           
Pf       cluster_guilt           1.0       0.0  0.277778  0.471831  0.405517
         feature_knn             1.0  0.444444  0.671429  0.609091  0.599626
         graph_convolution       1.0       0.6    0.6125    0.6325    0.5935
         layer_propagation      0.75  0.111111  0.168831  0.158311  0.177402
         layer_vote              1.0      0.45     0.625    0.5575      0.54
         map_neighbours         0.75  0.315789  0.441176  0.431818  0.465023
         physical_partners       NaN       NaN  0.333333  0.956522  0.852941
         random_forest          0.75       0.5      0.75    0.6475    0.6525
         stacking                1.0      0.65    0.6125    0.6175    0.5975
         stratum_focus           1.0  0.444444  0.671429  0.609091  0.599626
         structural_homology     NaN       0.5       0.5  0.481481  0.459658
         supervised_classifier   1.0      0.55    0.6125     0.605     0.582
         triangulation           1.0  0.533333  0.730159  0.668693  0.698076
         understudied_first      1.0  0.533333  0.730159  0.668693  0.698076
Tg       feature_knn             0.5    0.4375  0.557377  0.459119  0.504692
         layer_propagation      0.25  0.333333  0.142857  0.173913  0.165848
         layer_vote             0.25       0.4    0.5125      0.42     0.446
         physical_partners       0.0       0.4  0.541667  0.539568  0.611111
         random_forest          0.75      0.55     0.625     0.525    0.5415
         stratum_focus           0.5    0.4375  0.557377  0.459119  0.504692

## Whole classes hidden

Every gene of one class hidden at once. `right` is zero by construction: with every example of a label hidden, no strategy that learns from examples can name it. What is measured is `together` -- the share given one and the same label -- i.e. whether the strategy still recognises them as one thing, and `placed_at`, where it put them. Only sets with five or more answered members are ranked.

In [6]:
cls = led[(led['mode'] == 'set') & ~led['set_name'].astype(str).str.startswith('random')]
w = T.set_summary(cls)
assert (w['right'] == 0).all(), 'a hidden label was predicted: the class was not fully hidden'
w = w[w['answered'] >= 5]
w[['organism', 'strategy', 'set_name', 'set_size', 'answered', 'together', 'placed_at']].sort_values('together', ascending=False).head(20).round(2)

,organism,strategy,set_name,set_size,answered,together,placed_at
270,Tg,random_forest,19S proteasome,17,17,1,cytosol
42,Tg,feature_knn,19S proteasome,17,17,1,cytosol
40,Pf,cluster_guilt,ribosomes,65,56,1,cytosol
330,Tg,stratum_focus,19S proteasome,17,17,1,cytosol
3,Tg,cluster_guilt,60S ribosome,40,12,1,40S ribosome
144,Tg,layer_vote,19S proteasome,17,17,1,cytosol
414,Tg,supervised_classifier,19S proteasome,17,17,1,cytosol
417,Tg,supervised_classifier,60S ribosome,40,40,0.98,"40S ribosome, cytosol"
91,Pf,graph_convolution,erythrocyte membrane,42,42,0.95,"Maurer's cleft, apicoplast"
437,Pf,supervised_classifier,erythrocyte membrane,42,42,0.95,"Maurer's cleft, apicoplast"


## Your own list

`T.my_list(ctx, genes, target)` hides a pasted list together and asks every strategy. Two real labelled genes, as an example:

In [7]:
from starplast import strategies as S
ctx = S.Context.shipped('Tg')
rows = T.my_list(ctx, ['TGME49_294550', 'TGME49_244470'], 'compartment', strategies=['feature_knn', 'physical_partners', 'layer_vote'])
rows[['strategy', 'gene_id', 'truth', 'prediction', 'correct', 'abstained']]

,strategy,gene_id,truth,prediction,correct,abstained
0,feature_knn,TGME49_294550,nucleus - chromatin,nucleus - chromatin,True,False
1,feature_knn,TGME49_244470,apical 1,NaN,None,True
2,physical_partners,TGME49_294550,nucleus - chromatin,nucleus - chromatin,True,False
3,physical_partners,TGME49_244470,apical 1,apical 1,True,False
4,layer_vote,TGME49_294550,nucleus - chromatin,nucleus - chromatin,True,False
5,layer_vote,TGME49_244470,apical 1,apical 1,True,False
